In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [17]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [5]:
# Load the dataset
housing = fetch_california_housing(as_frame=True)

X = housing.data
y = housing.target

print(X.head())
print(y.head())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64


In [9]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3,random_state=42)

In [12]:
# Train the model
gbr.fit(X_train, y_train)

GradientBoostingRegressor(random_state=42)

In [14]:
y_pred = gbr.predict(X_test)

print(y_pred[:10])

[0.50518761 1.09334601 4.24570956 2.54517359 2.27910301 1.72390996
 2.33719663 1.71977379 3.09962388 4.30446906]


In [15]:
# Let's compare the values
comparison = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
})

print(comparison)

    Actual  Predicted
0  0.47700   0.505188
1  0.45800   1.093346
2  5.00001   4.245710
3  2.18600   2.545174
4  2.78000   2.279103
5  1.58700   1.723910
6  1.98200   2.337197
7  1.57500   1.719774
8  3.40000   3.099624
9  4.46600   4.304469


In [18]:
# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 0.3716425690425596
MSE : 0.2939973248643864
RMSE: 0.5422152016168362
R²  : 0.7756446042829697


## 7. The important part: what sklearn actually built

#### Remember our conceptual model:

```Initial prediction
      ↓
Tree 1
      ↓
Residuals
      ↓
Tree 2
      ↓
New residuals
      ↓
Tree 3
      ↓
...
      ↓
Tree 100
      ↓
Final prediction

In [25]:
print(len(gbr.estimators_))
print(gbr.estimators_[0])

100
[DecisionTreeRegressor(criterion='friedman_mse', max_depth=3,
                       random_state=RandomState(MT19937) at 0x18F04E1FA40)]


In [27]:
staged_predictions = gbr.staged_predict(X_test)

predictions = []

for i, pred in enumerate(staged_predictions, start=1):
    if i in [1, 5, 10, 25, 50, 100]:
        predictions.append({
            "Trees": i,
            "RMSE": mean_squared_error(y_test, pred) ** 0.5,
            "R2": r2_score(y_test, pred)
        })

pd.DataFrame(predictions)

,Trees,RMSE,R2
0,1,1.087157,0.098061
1,5,0.920484,0.353416
2,10,0.808273,0.501449
3,25,0.660509,0.667071
4,50,0.579794,0.743469
5,100,0.542215,0.775645


In [30]:
# Try predicting one new house
new_house = X_test.iloc[[0]]

prediction = gbr.predict(new_house)

print("Prediction:", prediction[0])
print("Actual:", y_test.iloc[0])

Prediction: 0.5051876120863898
Actual: 0.477


## Conceptually:

```new_house
    │
    ▼
Tree 1 ──► contribution
    │
    ▼
Tree 2 ──► correction
    │
    ▼
Tree 3 ──► correction
    │
    ▼
 ...
    │
    ▼
Tree 100 ─► correction
    │
    ▼
FINAL PREDICTION